In [ ]:
import os
import pandas as pd
from entsoe import EntsoePandasClient
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("ENTSOE_API_TOKEN") or os.getenv("ENTSOE_API_KEY")
if not api_key:
    raise ValueError("Missing ENTSOE_API_TOKEN / ENTSOE_API_KEY in environment")

client = EntsoePandasClient(api_key=api_key)
print("ENTSO-E client initialized.")

In [ ]:
# Test 1: Generation Actuals (1-month chunk)
start = pd.Timestamp("2024-01-01T00:00:00Z")
end = pd.Timestamp("2024-02-01T00:00:00Z")

country_code = "10Y1001A1001A83F"  # DE physical/control area

gen = client.query_generation(country_code=country_code, start=start, end=end)
print("Generation shape:", gen.shape)
display(gen.head())

In [ ]:
# Test 2: Wind/Solar forecasts (Day-Ahead and Intraday)
start = pd.Timestamp("2024-01-01T00:00:00Z")
end = pd.Timestamp("2024-02-01T00:00:00Z")
country_code = "10Y1001A1001A83F"

forecast_da = client.query_wind_and_solar_forecast(
    country_code=country_code,
    start=start,
    end=end,
    process_type="A01",  # day-ahead
)
print("DA forecast shape:", forecast_da.shape)
display(forecast_da.head())

try:
    forecast_id = client.query_wind_and_solar_forecast(
        country_code=country_code,
        start=start,
        end=end,
        process_type="A18",  # intraday
    )
except Exception:
    forecast_id = client.query_wind_and_solar_forecast(
        country_code=country_code,
        start=start,
        end=end,
        process_type="A40",  # fallback intraday/current
    )

print("ID forecast shape:", forecast_id.shape)
display(forecast_id.head())

In [ ]:
# Data inspection: columns + null counts for physical ENTSO-E series
for name, df_ in {
    "generation": gen,
    "forecast_da": forecast_da,
    "forecast_id": forecast_id,
}.items():
    print(f"\n=== {name} ===")
    print("Columns:", list(df_.columns))
    print("Null counts:")
    print(df_.isna().sum().sort_values(ascending=False).head(10))